# Advanced Resume Parsing using Natural Language Processing

This notebook demonstrates the complete NLP pipeline for parsing resumes:
1. Text extraction and preprocessing
2. Feature extraction (TF-IDF, Word2Vec, hand-crafted features)
3. Section classification model training and evaluation
4. End-to-end resume parsing

## 1. Setup & Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    clean_text, preprocess_text, extract_contact_info, identify_sections
)
from src.feature_extraction import TFIDFExtractor, HandcraftedFeatureExtractor
from src.model import ResumeParser, SkillsExtractor, extract_education
from src.evaluation import compute_metrics, print_evaluation_summary, plot_confusion_matrix

print('Imports successful!')

## 2. Load Sample Resumes

In [ ]:
RESUME_DIR = '../data/sample_resumes'

resumes = {}
for fname in os.listdir(RESUME_DIR):
    if fname.endswith('.txt'):
        path = os.path.join(RESUME_DIR, fname)
        with open(path, encoding='utf-8') as f:
            resumes[fname] = f.read()
        print(f'Loaded {fname}: {len(resumes[fname])} chars')

print(f'\nTotal resumes loaded: {len(resumes)}')

## 3. Text Preprocessing

In [ ]:
# Demonstrate preprocessing on the first resume
first_key = list(resumes.keys())[0]
raw_text = resumes[first_key]

print('=== RAW TEXT (first 400 chars) ===')
print(raw_text[:400])

cleaned = clean_text(raw_text)
print('\n=== CLEANED TEXT (first 400 chars) ===')
print(cleaned[:400])

In [ ]:
# Tokenisation and preprocessing
tokens = preprocess_text(raw_text, lowercase=True, remove_punct=True,
                          remove_stops=True, do_lemmatize=False)
print(f'Token count (after preprocessing): {len(tokens)}')
print('Sample tokens:', tokens[:20])

## 4. Contact Information Extraction

In [ ]:
for fname, text in resumes.items():
    contact = extract_contact_info(text)
    print(f'\n{fname}:')
    for k, v in contact.items():
        if v:
            print(f'  {k}: {v}')

## 5. Section Identification

In [ ]:
sections = identify_sections(cleaned)
print('Identified sections:')
for section, content in sections.items():
    print(f'  [{section}] ({len(content)} chars): {content[:80].strip()}...')

## 6. Feature Extraction

In [ ]:
# TF-IDF features
corpus = list(resumes.values())

tfidf = TFIDFExtractor(max_features=100)
tfidf_matrix = tfidf.fit_transform(corpus)
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

# Show top terms
vocab = tfidf.vocabulary
top_terms = sorted(vocab.items(), key=lambda x: x[1])[:20]
print('Sample vocabulary terms:', [t[0] for t in top_terms])

In [ ]:
# Hand-crafted features
hc_extractor = HandcraftedFeatureExtractor()
hc_matrix = hc_extractor.to_matrix(corpus)
feature_names = hc_extractor.feature_names

print('Hand-crafted feature matrix shape:', hc_matrix.shape)
print('Feature names:', feature_names)

df_features = pd.DataFrame(hc_matrix, columns=feature_names,
                            index=[k.replace('.txt','') for k in resumes.keys()])
df_features

## 7. Skills Extraction

In [ ]:
skills_extractor = SkillsExtractor()

for fname, text in resumes.items():
    skills = skills_extractor.extract(text)
    print(f'\n{fname}:')
    print(f'  Technical skills ({len(skills["technical"])}): {skills["technical"][:10]}')
    print(f'  Soft skills ({len(skills["soft"])}): {skills["soft"][:5]}')

## 8. Education Extraction

In [ ]:
for fname, text in resumes.items():
    sections_data = identify_sections(text)
    edu_text = sections_data.get('education', '')
    entries = extract_education(edu_text)
    print(f'\n{fname} - Education entries:')
    for entry in entries:
        print(f'  Degree: {entry["degree"]} | Institution: {entry["institution"]}')
        print(f'  Duration: {entry["start_year"]} – {entry["end_year"]} | GPA: {entry["gpa"]}')

## 9. Section Classifier – Synthetic Training Data

We create a small synthetic dataset to demonstrate training and evaluating
a section classifier.

In [ ]:
from src.model import SectionClassifier
from sklearn.model_selection import train_test_split

# Synthetic training samples per section
synthetic_data = [
    # education
    ('B.Tech Computer Science MIT 2018 GPA 3.8', 'education'),
    ('M.S. Data Science Stanford University 2020', 'education'),
    ('Bachelor of Science Mathematics UCLA 2016', 'education'),
    ('Ph.D. Artificial Intelligence CMU 2022 dissertation NLP', 'education'),
    ('Associate degree Information Technology 2015', 'education'),
    # experience
    ('Software Engineer Google 2019 2022 built APIs Python', 'experience'),
    ('Data Scientist Amazon 2020 machine learning models deployment', 'experience'),
    ('Junior Developer startup 2017 web applications React Node', 'experience'),
    ('Senior Analyst consulting firm 2021 business intelligence SQL', 'experience'),
    ('Product Manager fintech company 2018 roadmap stakeholders', 'experience'),
    # skills
    ('Python Java JavaScript machine learning NLP scikit-learn', 'skills'),
    ('React Node.js TypeScript Docker Kubernetes AWS', 'skills'),
    ('SQL PostgreSQL MongoDB data analysis pandas numpy', 'skills'),
    ('TensorFlow PyTorch deep learning computer vision', 'skills'),
    ('Leadership communication project management agile', 'skills'),
    # projects
    ('Built resume parser NLP spaCy 91% accuracy deployed Streamlit', 'projects'),
    ('Developed recommendation system collaborative filtering 88%', 'projects'),
    ('Created chatbot GPT-3 fine-tuning customer support', 'projects'),
    ('Designed distributed database 10000 transactions per second', 'projects'),
    ('Open source contribution pandas bug fix 200 stars GitHub', 'projects'),
    # certifications
    ('AWS Certified Solutions Architect 2022', 'certifications'),
    ('Google Professional Data Engineer certification', 'certifications'),
    ('Coursera Deep Learning Specialization certificate', 'certifications'),
    ('Microsoft Azure Developer Associate 2021', 'certifications'),
    ('PMI PMP certification project management professional', 'certifications'),
    # summary
    ('Results-driven engineer 5 years experience scalable systems', 'summary'),
    ('Passionate data scientist specialising NLP machine learning', 'summary'),
    ('Seeking challenging role leverage software development skills', 'summary'),
    ('Experienced manager leading cross-functional engineering teams', 'summary'),
    ('Recent graduate eager to apply machine learning real world', 'summary'),
]

texts = [d[0] for d in synthetic_data]
labels = [d[1] for d in synthetic_data]

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

In [ ]:
# Train section classifier
clf = SectionClassifier(model_type='lr')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print_evaluation_summary(y_test, y_pred)

# Confusion matrix
section_labels = sorted(set(y_test + y_pred))
plot_confusion_matrix(y_test, y_pred, labels=section_labels,
                       title='Section Classifier – Confusion Matrix')

## 10. End-to-End Resume Parsing Demo

In [ ]:
import json

parser = ResumeParser(use_ner=False)  # set use_ner=True if spaCy model is installed

for fname, text in resumes.items():
    result = parser.parse(text)
    print(f'\n{'='*60}')
    print(f'Resume: {fname}')
    print(f'{'='*60}')
    print(f'Contact: {result["contact"]}')
    print(f'Sections: {list(result["sections"].keys())}')
    print(f'Technical skills: {result["skills"]["technical"][:8]}')
    print(f'Education entries: {len(result["education"])}')

## Summary

This notebook demonstrated:
- **Text preprocessing**: cleaning, tokenisation, stopword removal
- **Contact extraction**: email, phone, LinkedIn, GitHub using regex
- **Section identification**: rule-based section splitting
- **TF-IDF feature extraction** using scikit-learn
- **Hand-crafted features**: boolean flags and counts
- **Skills extraction**: keyword matching against a curated skills database
- **Education extraction**: rule-based degree and year parsing
- **Section classification**: Logistic Regression on TF-IDF features
- **Evaluation**: Accuracy, Precision, Recall, F1, Confusion Matrix
- **End-to-end parsing** with the `ResumeParser` class

For a live interactive demo, run `streamlit run app.py` from the project root.